In [12]:
# 1. Load package and configure paths-----------
# Verify loaded dada2 version

library(dada2)
dada2_version <- packageVersion("dada2")

In [13]:
# Define main project base path and working directories
base_path <- "/home/tatiana/PROYECTOS/repository/sequences" # change by your path
its_dir <- file.path(base_path, "its_results", "itsxpress")
filt_dir <- file.path(base_path, "dada2_filtered")

In [14]:
# Create the filtered output directory if it does not exist
if (!dir.exists(filt_dir)) {
    dir.create(filt_dir, recursive = TRUE)
}

In [15]:
# 2. Detect FASTQ files from the ITSxpress step-----------

# Get sorted lists of R1 and R2 files processed by itsxpress
itsFs <- sort(list.files(its_dir, pattern = "_R1_001.fastq.gz", full.names = TRUE))
itsRs <- sort(list.files(its_dir, pattern = "_R2_001.fastq.gz", full.names = TRUE))

In [16]:
# 3. Define output paths for filtered files -------------------------
# Set matching paths for the upcoming filtered output files
filtFs <- file.path(filt_dir, basename(itsFs))
filtRs <- file.path(filt_dir, basename(itsRs))

In [17]:
# 4. Check that the input directory and file pairing are valid -------
stopifnot(dir.exists(its_dir))

# Check that forward and reverse files are paired
stopifnot(
    length(itsFs) > 0,
    length(itsFs) == length(itsRs)
)

In [18]:
# 5. Quality filter for the variable length of ITS -------------------
# Note that 'truncLen' is not used due to ITS length variability
track_dada2 <- filterAndTrim(
    fwd = itsFs, 
    filt = filtFs,
    rev = itsRs, 
    filt.rev = filtRs,
    maxN = 0,               # excluding N bases
    maxEE = c(2, 5),        # c(R1, R2): max. errors expected for MiSeq 2x300
    truncQ = 2,             # Cut the read if quality score drops to Phred
    minLen = 50,            # Discard sequences <50 bp
    rm.phix = TRUE,         # Remove PhiX control sequences
    compress = TRUE,        # Save the obtained files in .gz
    multithread = TRUE      # Use all processor threads to speed up calculation
)

In [19]:
# 6. Tracking the reads after this step -----------------------------
# Return the tracking matrix visually without explicit print()
track_dada2

,reads.in,reads.out
its-LC1_S5_L001_R1_001.fastq.gz,343470,309244
its-LC2_S7_L001_R1_001.fastq.gz,672440,620623
its-LC3_S9_L001_R1_001.fastq.gz,214970,199301
its-LL1_S2_L001_R1_001.fastq.gz,288514,259862
its-LL2_S3_L001_R1_001.fastq.gz,175646,163839
its-LL3_S4_L001_R1_001.fastq.gz,313888,292860
its-LRS1_S6_L001_R1_001.fastq.gz,51131,46447
its-LRS2_S8_L001_R1_001.fastq.gz,148864,134738
its-LRS3_S10_L001_R1_001.fastq.gz,969025,867819
its-negC_S1_L001_R1_001.fastq.gz,215,189


In [ ]:
# 7. Learn error models ---------------------------------------------
set.seed(100)

errF <- learnErrors(
    filtFs,
    multithread = TRUE,
    verbose = TRUE
)

errR <- learnErrors(
    filtRs,
    multithread = TRUE,
    verbose = TRUE
)

In [ ]:
# 8. Dereplication & ASVs inference ---------------------------------
# Dereplicate filtered fastq files (required for proper merging later)
derepFs <- derepFastq(filtFs, verbose = TRUE)
derepRs <- derepFastq(filtRs, verbose = TRUE)

# Infer real variants in forward reads
dadaFs <- dada(
    derepFs,
    err = errF,
    multithread = TRUE,
    verbose = TRUE
)

# Infer real variants in reverse reads
dadaRs <- dada(
    derepRs,
    err = errR,
    multithread = TRUE,
    verbose = TRUE
)

In [ ]:
# 9. Merge pairs step -----------------------------------------------
# Merge forward and reverse reads
mergers <- mergePairs(
    dadaF = dadaFs, 
    derepF = filtFs,     # Nota: DADA2 requiere habitualmente un objeto derep, asegúrate de haberlo ejecutado antes si fuese necesario
    dadaR = dadaRs, 
    derepR = filtRs, 
    minOverlap = 12,     # Minimum 12 matching bases to validate alignment
    maxMismatch = 0,     # No misalignments permitted in the overlap region
    verbose = TRUE
)

# Inspect the merged reads from the first sample visually
head(mergers[[1]])

In [ ]:
# Construct sequence table (ASV table)
seqtab <- makeSequenceTable(mergers)

In [ ]:
# # 10. Remove chimeras ----------------------------------------------
seqtab.nochim <- removeBimeraDenovo(
    seqtab, 
    method = "consensus", 
    multithread = TRUE, 
    verbose = TRUE
)

In [ ]:
# # 11) Assign taxonomy with UNITE database --------------------------
unite_fasta <- "/home/tatiana/PROYECTOS/database/sh_general_release_dynamic_19.02.2025.fasta" # download the database

# Run Bayesian classifier
taxa <- assignTaxonomy(
    seqs = seqtab.nochim, 
    refFasta = unite_fasta, 
    tryRC = TRUE,            # Mandatory for ITS region (reverse complement check)
    multithread = TRUE,      # Use all processor threads
    verbose = TRUE
)




In [ ]:
# # 12. Format and export ASV and Taxonomy tables for Python ---------
# Filter out negative and positive control samples from the ASV table
asv_table_orig <- seqtab.nochim
columnas_biologicas_asv <- grep("negC|posC", rownames(asv_table_orig), invert = TRUE, value = TRUE)
asv_table_filt <- asv_table_orig[columnas_biologicas_asv, , drop = FALSE]

# Export filtered Feature Table (Samples as rows, ASV sequences as columns)
write.csv(
    asv_table_filt, 
    file.path(base_path, "feature_table.csv"), 
    quote = FALSE
)

# Format taxonomy table with standard prefixes (k__, p__, c__, etc.)
tax_only <- as.data.frame(taxa, stringsAsFactors = FALSE)
asv_sequences_reales <- rownames(tax_only)

prefijos <- c(Kingdom = "k__", Phylum = "p__", Class = "c__", Order = "o__", Family = "f__", Genus = "g__", Species = "s__")

for (col_name in colnames(tax_only)) {
    tax_only[is.na(tax_only[, col_name]) | tax_only[, col_name] == "" | tax_only[, col_name] == "NA", col_name] <- "NA"
    
    if (col_name %in% names(prefijos)) {
        pref <- prefijos[col_name]
        tax_only[, col_name] <- gsub("^k__|^p__|^c__|^o__|^f__|^g__|^s__", "", tax_only[, col_name])
        tax_only[, col_name] <- ifelse(tax_only[, col_name] == "NA", "NA", paste0(pref, tax_only[, col_name]))
    }
}

# Assemble final taxonomy dataframe with sequences as a physical column
taxonomy_asv_final <- data.frame(
    Sequence = asv_sequences_reales,
    tax_only,
    stringsAsFactors = FALSE
)

# Export taxonomy table CSV
write.csv(
    taxonomy_asv_final, 
    file.path(base_path, "taxonomy_table.csv"), 
    row.names = FALSE, 
    quote = FALSE
)

# Check linkage alignment between Feature Table and Taxonomy
linkage_check <- all(colnames(asv_table_filt) == taxonomy_asv_final$Sequence)

# Return final verification status visually
list(
    total_filtered_samples = nrow(asv_table_filt),
    total_asvs = ncol(asv_table_filt),
    tables_correctly_linked = linkage_check
)

In [ ]:
# # 13. Pipeline complete read-tracking table -----------------------

library(ShortRead)

# Locate raw and Cutadapt FASTQ files (search only in specified directories)
raw_dir <- base_path
cut_dir <- file.path(base_path, "cutadapt")

raw_files <- sort(list.files(
    raw_dir,
    pattern = "_R1_001.fastq.gz$",
    full.names = TRUE,
    recursive = FALSE
))

cut_files <- sort(list.files(
    cut_dir,
    pattern = "_R1_001.fastq.gz$",
    full.names = TRUE,
    recursive = FALSE
))

# Validate that FASTQ files exist
if (length(raw_files) == 0) {
    stop("No raw R1 FASTQ files were found. Check raw_dir.")
}

if (length(cut_files) == 0) {
    stop("No Cutadapt R1 FASTQ files were found. Check cut_dir.")
}

if (length(itsFs) == 0) {
    stop("No ITSxpress R1 FASTQ files were found. Check itsFs.")
}

# Function to count reads in FASTQ files
count_fastq <- function(fastq_path) {
    if (length(fastq_path) == 0 || !file.exists(fastq_path)) {
        stop(paste("FASTQ file not found:", fastq_path))
    }
    fq_data <- ShortRead::readFastq(fastq_path)
    length(ShortRead::sread(fq_data))
}

# Count reads at the raw and Cutadapt stages
reads_raw <- sapply(raw_files, count_fastq)
reads_cut <- sapply(cut_files, count_fastq)

# Helper function to extract sample identifiers from filenames
get_sample_ids <- function(files) {
    sub("_R1_001.fastq.gz$", "", basename(files))
}

raw_samples <- get_sample_ids(raw_files)
cut_samples <- get_sample_ids(cut_files)
its_samples <- get_sample_ids(itsFs)

# Verify sample correspondence and uniqueness across stages
if (anyDuplicated(raw_samples) || anyDuplicated(cut_samples) || anyDuplicated(its_samples)) {
    stop("Duplicate sample identifiers were found in the FASTQ file lists.")
}

if (!setequal(raw_samples, cut_samples) || !setequal(raw_samples, its_samples)) {
    stop("Sample identifiers do not match across raw, Cutadapt, and ITSxpress files. Check file names.")
}

# Reorder counts to match the ITSxpress sample order
reads_raw <- reads_raw[match(its_samples, raw_samples)]
reads_cut <- reads_cut[match(its_samples, cut_samples)]

if (anyNA(reads_raw) || anyNA(reads_cut)) {
    stop("Some samples could not be matched across processing stages.")
}

# Validate sample length consistency before building tracking table
if (length(itsFs) != length(dadaFs) ||
    length(itsFs) != length(mergers) ||
    length(itsFs) != nrow(seqtab.nochim)) {
    stop("The number of samples differs between ITSxpress files, DADA2 inference, merged reads, and final sequence

In [ ]:
 # 13. Pipeline complete read-tracking table -----------------------

library(ShortRead)

# Locate raw and Cutadapt FASTQ files (search only in specified directories)
raw_dir <- base_path
cut_dir <- file.path(base_path, "its_results", "cutadapt")
raw_files <- sort(list.files(
    raw_dir,
    pattern = "_R1_001.fastq.gz$",
    full.names = TRUE,
    recursive = FALSE
))

cut_files <- sort(list.files(
    cut_dir,
    pattern = "_R1_001.fastq.gz$",
    full.names = TRUE,
    recursive = FALSE
))

# Validate that FASTQ files exist
if (length(raw_files) == 0) {
    stop("No raw R1 FASTQ files were found. Check raw_dir.")
}

if (length(cut_files) == 0) {
    stop("No Cutadapt R1 FASTQ files were found. Check cut_dir.")
}

if (length(itsFs) == 0) {
    stop("No ITSxpress R1 FASTQ files were found. Check itsFs.")
}

# Function to count reads in FASTQ files
count_fastq <- function(fastq_path) {
    if (length(fastq_path) == 0 || !file.exists(fastq_path)) {
        stop(paste("FASTQ file not found:", fastq_path))
    }
    fq_data <- ShortRead::readFastq(fastq_path)
    length(ShortRead::sread(fq_data))
}

# Count reads at the raw and Cutadapt stages
reads_raw <- sapply(raw_files, count_fastq)
reads_cut <- sapply(cut_files, count_fastq)

# Helper function to extract sample identifiers from filenames
get_sample_ids <- function(files) {
    sub("_R1_001.fastq.gz$", "", basename(files))
}

raw_samples <- get_sample_ids(raw_files)
cut_samples <- get_sample_ids(cut_files)
its_samples <- get_sample_ids(itsFs)

# Verify sample correspondence and uniqueness across stages
if (anyDuplicated(raw_samples) || anyDuplicated(cut_samples) || anyDuplicated(its_samples)) {
    stop("Duplicate sample identifiers were found in the FASTQ file lists.")
}

if (!setequal(raw_samples, cut_samples) || !setequal(raw_samples, its_samples)) {
    stop("Sample identifiers do not match across raw, Cutadapt, and ITSxpress files. Check file names.")
}

# Reorder counts to match the ITSxpress sample order
reads_raw <- reads_raw[match(its_samples, raw_samples)]
reads_cut <- reads_cut[match(its_samples, cut_samples)]

if (anyNA(reads_raw) || anyNA(reads_cut)) {
    stop("Some samples could not be matched across processing stages.")
}

# Validate sample length consistency before building tracking table
if (length(itsFs) != length(dadaFs) ||
    length(itsFs) != length(mergers) ||
    length(itsFs) != nrow(seqtab.nochim)) {
    stop("The number of samples differs between ITSxpress files, DADA2 inference, merged reads, and final sequence table.")
}

# Extract read counts from DADA2 objects helper
getN <- function(x) sum(dada2::getUniques(x))

# Build the complete tracking dataframe
track_complete <- data.frame(
    Sample = its_samples,
    Raw_Reads = as.numeric(reads_raw),
    Post_Cutadapt = as.numeric(reads_cut),
    Post_ITSxpress = as.numeric(track_dada2[, "reads.in"]),
    DADA2_Filtered = as.numeric(track_dada2[, "reads.out"]),
    DADA2_Inferred = as.numeric(sapply(dadaFs, getN)),
    Merged_Reads = as.numeric(sapply(mergers, getN)),
    Final_NonChimeric_Reads = as.numeric(rowSums(seqtab.nochim)),
    row.names = NULL
)

# Export the tracking table to CSV
tracking_csv_path <- file.path(base_path, "pipeline_tracking_completo.csv")
write.csv(track_completo, tracking_csv_path, row.names = FALSE, quote = FALSE)

# Return the complete tracking table visually at the end of the cell
track_complete